# Fig. 4B: blood APA signature volcano (Delta-psi)
Primary gene-level DRIMSeq DTU (`~Batch+Diagnosis`), 81 genes at adj-p<0.05. The effect-size axis is Delta-psi (change in W = rank-weighted PAS position), as in the other figures; it is robust to long-UTR-isoform leverage and is used instead of the bp-weighted mean-length axis (the two agree on sign for 93% of hits). Confound sensitivity: fig. S4D; within-family tests: Fig. 4D to 4G and `analysis/blood/07-260615-within_family_systematic_apa.ipynb`.

## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0,f"{REPO_ROOT}/figures/style"); import paper_fig_style as S; S.apply_rcparams()
PANEL="fig5b_dtu_volcano"; B=f"{DATA_ROOT}/3utr/Blood/2025_analyses/DTU_curated_sexadj/"
g=pd.read_csv(B+"dtu_unadj_gene.csv"); f=pd.read_csv(B+"dtu_unadj_feature.csv")
f["symbol"]=f.feature_id.astype(str).str.split("-").str[0].str.split("_").str[0]
g["symbol"]=g.gene_id.map(f.groupby("gene_id")["symbol"].agg(lambda s:s.mode().iat[0] if len(s.mode()) else s.iloc[0]))
dp=pd.read_csv(B+"gene_dpsi_curated.csv"); g=g.merge(dp,on="gene_id",how="left")   # Delta-psi (the effect size used throughout the paper)
g["nl"]=-np.log10(g.adj_pvalue.clip(lower=1e-300)); g["sig"]=g.adj_pvalue<0.05
print("hits adj<0.05:",int(g.sig.sum()),"| Dpsi range",round(g.dpsi.min(),3),round(g.dpsi.max(),3))


In [ ]:
fig,ax=plt.subplots(figsize=(S.HALF_W_PX/300, S.QUARTER_H_PX/300))
ax.scatter(g.loc[~g.sig,"dpsi"],g.loc[~g.sig,"nl"],s=5,c="#cccccc",alpha=0.6,rasterized=True,edgecolors="none")
ax.scatter(g.loc[g.sig,"dpsi"],g.loc[g.sig,"nl"],s=14,c="#2c7fb8",alpha=0.85,edgecolors="none",label=f"adj-p<0.05 (n={int(g.sig.sum())})")
ax.axhline(-np.log10(0.05),ls="--",c="grey",lw=0.6); ax.axvline(0,ls=":",c="grey",lw=0.5)
try:
    from adjustText import adjust_text
    t=[ax.text(r.dpsi,r.nl,r["symbol"],fontsize=S.FONT_ANNOT-0.5,fontstyle="italic") for _,r in g[g.sig].sort_values("adj_pvalue").head(8).iterrows()]
    adjust_text(t,ax=ax,arrowprops=dict(arrowstyle="-",color="gray",lw=0.3))
except Exception: pass
ax.set_xlabel(r"$\Delta\psi$ (ASD - CTRL; + = lengthening)",fontsize=S.FONT_LABEL); ax.set_ylabel(r"$-\log_{10}$ adj-p (primary)",fontsize=S.FONT_LABEL)
ax.legend(fontsize=S.FONT_LEGEND,frameon=False,loc="upper right"); ax.tick_params(labelsize=S.FONT_TICK); [s.set_visible(False) for s in (ax.spines["top"],ax.spines["right"])]
fig.subplots_adjust(left=0.16,right=0.96,bottom=0.14,top=0.96)
S.save_and_show(fig,PANEL,figure="Fig5")


### Eighth-height version of the blood DTU volcano

`fig5b_dtu_volcano__eighth` at `HALF_W x EIGHTH_H` (fewer gene labels for the short panel).

In [ ]:
# --- Eighth-height version of the blood DTU volcano (reuses `g`) ---
fig, ax = plt.subplots(figsize=(S.HALF_W_IN, S.EIGHTH_H_IN))
ax.scatter(g.loc[~g.sig, "dpsi"], g.loc[~g.sig, "nl"], s=4, c="#cccccc", alpha=0.6, rasterized=True, edgecolors="none")
ax.scatter(g.loc[g.sig, "dpsi"], g.loc[g.sig, "nl"], s=9, c="#2c7fb8", alpha=0.85, edgecolors="none",
           label=f"adj-p<0.05 (n={int(g.sig.sum())})")
ax.axhline(-np.log10(0.05), ls="--", c="grey", lw=0.6); ax.axvline(0, ls=":", c="grey", lw=0.5)
try:
    from adjustText import adjust_text
    t = [ax.text(r.dpsi, r.nl, r["symbol"], fontsize=S.FONT_ANNOT - 1, fontstyle="italic")
         for _, r in g[g.sig].sort_values("adj_pvalue").head(4).iterrows()]
    adjust_text(t, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3))
except Exception:
    pass
ax.set_xlabel(r"$\Delta\psi$ (ASD - CTRL; + = lengthening)", fontsize=S.FONT_TICK)
ax.set_ylabel(r"$-\log_{10}$ adj-p", fontsize=S.FONT_TICK)
ax.legend(fontsize=S.FONT_LEGEND - 1, frameon=False, loc="upper right", handletextpad=0.2, borderpad=0.1)
ax.tick_params(labelsize=S.FONT_TICK - 1); [s.set_visible(False) for s in (ax.spines["top"], ax.spines["right"])]
fig.subplots_adjust(left=0.15, right=0.97, bottom=0.30, top=0.95)
S.save_and_show(fig, "fig5b_dtu_volcano__eighth", figure="Fig5")
